# HR Management Portal
## End-to-End Portal Design (Assignment 3)

This notebook implements a complete **HR Management Portal** using core Python concepts as required by the assignment.

### Features
- **Employee Portal**: Request leaves, upload documents, view announcements
- **HR Portal**: Approve/reject leaves, post announcements, view documents, generate reports, add employees
- Persistent storage using JSON files
- Robust error handling
- Interactive console interface + Streamlit frontend (see `app.py`)

### Assignment Requirements Covered
| Requirement | Implementation |
|-------------|----------------|
| Modular programming (`def`) | All core features in well-documented functions with parameters and returns |
| Loops (`for`, `while`) | `while True` main menu loop + `for` loops for searching/displaying data |
| File handling | JSON read/write on startup and updates |
| Exception handling | `try-except-finally` for file and input errors |
| Documentation | This notebook + README |

**Demo Credentials:**
- HR: `hr_admin` / `admin123`
- Employee: `emp_ali` / `emp123`


In [ ]:
import json
import os
from datetime import datetime

# Create directories
os.makedirs("data", exist_ok=True)
os.makedirs("docs", exist_ok=True)

# File paths
EMPLOYEES_FILE = "data/employees.json"
LEAVES_FILE = "data/leaves.json"
DOCUMENTS_FILE = "data/documents.json"
ANNOUNCEMENTS_FILE = "data/announcements.json"

print("✅ Setup complete. Data directories ready.")


## 1. Data Persistence Layer
Robust file handling with exception management.


In [ ]:
def load_data(filename):
    """Load JSON data. Handles FileNotFoundError and JSON errors."""
    try:
        if not os.path.exists(filename):
            return []
        with open(filename, 'r') as f:
            data = json.load(f)
        return data
    except FileNotFoundError:
        print(f"⚠️ File not found: {filename}. Starting with empty data.")
        return []
    except json.JSONDecodeError as e:
        print(f"❌ Corrupt JSON in {filename}: {e}")
        return []
    except PermissionError:
        print(f"❌ Permission denied: {filename}")
        return []
    except Exception as e:
        print(f"❌ Unexpected error loading {filename}: {e}")
        return []
    finally:
        pass  # Cleanup if needed

def save_data(filename, data):
    """Save data to JSON with error handling."""
    try:
        with open(filename, 'w') as f:
            json.dump(data, f, indent=4)
        return True
    except PermissionError:
        print(f"❌ Permission denied writing to {filename}")
        return False
    except Exception as e:
        print(f"❌ Error saving {filename}: {e}")
        return False
    finally:
        pass

print("✅ File handling functions defined.")


## 2. Sample Data Initialization
Creates demo users on first run.


In [ ]:
def initialize_sample_data():
    """Create sample employees if files don't exist."""
    if not os.path.exists(EMPLOYEES_FILE):
        sample_employees = [
            {
                "id": 1,
                "username": "hr_admin",
                "password": "admin123",
                "name": "Sarah Khan",
                "role": "HR",
                "department": "Human Resources",
                "email": "hr@company.com"
            },
            {
                "id": 2,
                "username": "emp_ali",
                "password": "emp123",
                "name": "Ali Ahmed",
                "role": "Employee",
                "department": "Engineering",
                "email": "ali@company.com"
            },
            {
                "id": 3,
                "username": "emp_sara",
                "password": "emp123",
                "name": "Sara Malik",
                "role": "Employee",
                "department": "Marketing",
                "email": "sara@company.com"
            }
        ]
        save_data(EMPLOYEES_FILE, sample_employees)
        print("✅ Sample employees created.")
    
    for f in [LEAVES_FILE, DOCUMENTS_FILE, ANNOUNCEMENTS_FILE]:
        if not os.path.exists(f):
            save_data(f, [])
            print(f"✅ Initialized {f}")

initialize_sample_data()
print("\nCurrent employees:")
for emp in load_data(EMPLOYEES_FILE):
    print(f"  - {emp['username']} ({emp['role']}): {emp['name']}")


## 3. Core Portal Functions
Modular functions as required: `user_login()`, `add_record()`, `search_record()`, `generate_report()`.


In [ ]:
def user_login(username, password):
    """
    Authenticate user.
    Parameters: username (str), password (str)
    Returns: user dict or None
    """
    employees = load_data(EMPLOYEES_FILE)
    for emp in employees:  # for loop for searching
        if emp["username"] == username and emp["password"] == password:
            return emp
    return None

def add_record(filename, record):
    """
    Add a new record to any data file.
    Auto-assigns ID. Returns True on success.
    """
    try:
        data = load_data(filename)
        record["id"] = max([r.get("id", 0) for r in data], default=0) + 1
        data.append(record)
        return save_data(filename, data)
    except Exception as e:
        print(f"❌ Error adding record: {e}")
        return False

def search_records(filename, key, value):
    """
    Search records by key-value pair.
    Uses for loop as required by assignment.
    Returns list of matching records.
    """
    data = load_data(filename)
    results = []
    for item in data:
        if str(item.get(key, "")).lower() == str(value).lower():
            results.append(item)
    return results

def generate_report(report_type="pending_leaves"):
    """
    Generate reports.
    Parameters: report_type (str)
    Returns: list of records
    """
    if report_type == "pending_leaves":
        leaves = load_data(LEAVES_FILE)
        pending = []
        for leave in leaves:
            if leave.get("status") == "Pending":
                pending.append(leave)
        return pending
    elif report_type == "all_documents":
        return load_data(DOCUMENTS_FILE)
    elif report_type == "announcements":
        return load_data(ANNOUNCEMENTS_FILE)
    else:
        return []

print("✅ Core functions defined: user_login, add_record, search_records, generate_report")


## 4. Feature Functions
Leave requests, document uploads, announcements.


In [ ]:
def request_leave(employee_id, employee_name, start_date, end_date, reason):
    """Employee leave request."""
    try:
        # Validate dates
        start = datetime.strptime(start_date, "%Y-%m-%d")
        end = datetime.strptime(end_date, "%Y-%m-%d")
        if start > end:
            print("❌ Start date cannot be after end date")
            return False
        
        record = {
            "employee_id": employee_id,
            "employee_name": employee_name,
            "start_date": start_date,
            "end_date": end_date,
            "reason": reason,
            "status": "Pending",
            "requested_on": datetime.now().strftime("%Y-%m-%d %H:%M")
        }
        return add_record(LEAVES_FILE, record)
    except ValueError as e:
        print(f"❌ Invalid date format: {e}")
        return False
    except Exception as e:
        print(f"❌ Error requesting leave: {e}")
        return False

def upload_document(employee_id, employee_name, filename, description):
    """Simulate document upload (stores metadata)."""
    record = {
        "employee_id": employee_id,
        "employee_name": employee_name,
        "filename": filename,
        "description": description,
        "uploaded_on": datetime.now().strftime("%Y-%m-%d %H:%M")
    }
    return add_record(DOCUMENTS_FILE, record)

def post_announcement(title, content, author):
    """HR posts announcement."""
    record = {
        "title": title,
        "content": content,
        "author": author,
        "date": datetime.now().strftime("%Y-%m-%d %H:%M")
    }
    return add_record(ANNOUNCEMENTS_FILE, record)

def update_leave_status(leave_id, new_status):
    """HR approves or rejects leave."""
    leaves = load_data(LEAVES_FILE)
    for leave in leaves:
        if leave["id"] == leave_id:
            leave["status"] = new_status
            return save_data(LEAVES_FILE, leaves)
    return False

print("✅ Feature functions defined.")


## 5. Sample Execution & Testing
Demonstrates the system in action (outputs shown below when run).


In [ ]:
# === DEMO 1: Login ===
print("=" * 50)
print("DEMO 1: User Login")
print("=" * 50)
user = user_login("hr_admin", "admin123")
if user:
    print(f"✅ Login successful: {user['name']} ({user['role']})")
else:
    print("❌ Login failed")

user2 = user_login("emp_ali", "wrongpass")
print(f"Wrong password test: {user2}")  # Should be None

# === DEMO 2: HR posts announcement ===
print("\n" + "=" * 50)
print("DEMO 2: Post Announcement")
print("=" * 50)
success = post_announcement(
    "Office Closed on Friday",
    "The office will be closed on Friday for maintenance. Work from home.",
    "Sarah Khan"
)
print(f"Announcement posted: {success}")

# === DEMO 3: Employee requests leave ===
print("\n" + "=" * 50)
print("DEMO 3: Leave Request")
print("=" * 50)
success = request_leave(
    employee_id=2,
    employee_name="Ali Ahmed",
    start_date="2026-11-20",
    end_date="2026-11-22",
    reason="Family event"
)
print(f"Leave requested: {success}")

# Invalid date test
request_leave(2, "Ali", "2026-11-25", "2026-11-20", "Test")  # Should fail

# === DEMO 4: Search and Report ===
print("\n" + "=" * 50)
print("DEMO 4: Search & Generate Report")
print("=" * 50)
pending = generate_report("pending_leaves")
print(f"Pending leaves: {len(pending)}")
for leave in pending:
    print(f"  - {leave['employee_name']}: {leave['start_date']} to {leave['end_date']} ({leave['status']})")

# === DEMO 5: Approve leave ===
print("\n" + "=" * 50)
print("DEMO 5: Approve Leave")
print("=" * 50)
if pending:
    update_leave_status(pending[0]["id"], "Approved")
    print("✅ Leave approved")
    print("Updated pending:", len(generate_report("pending_leaves")))

# === DEMO 6: Document upload ===
print("\n" + "=" * 50)
print("DEMO 6: Document Upload")
print("=" * 50)
upload_document(2, "Ali Ahmed", "resume.pdf", "Updated resume")
docs = search_records(DOCUMENTS_FILE, "employee_id", 2)
print(f"Ali's documents: {len(docs)}")
for d in docs:
    print(f"  - {d['filename']}: {d['description']}")

print("\n✅ All demos completed successfully!")


## 6. Interactive Console Portal
Full interactive menu using `while True` loop.

**Note:** Running the cell below will start the interactive portal.  
Use the demo credentials above. Type `0` to exit.


In [ ]:
def display_menu(role):
    """Display role-based menu."""
    print("\n" + "=" * 40)
    if role == "Employee":
        print("EMPLOYEE PORTAL")
        print("1. Request Leave")
        print("2. Upload Document")
        print("3. View My Leaves")
        print("4. View Announcements")
        print("5. Logout")
        print("0. Exit")
    else:
        print("HR PORTAL")
        print("1. Manage Leave Requests")
        print("2. Post Announcement")
        print("3. View All Documents")
        print("4. Generate Report")
        print("5. Add Employee")
        print("6. Logout")
        print("0. Exit")
    print("=" * 40)

def employee_menu(user):
    """Employee interactive menu."""
    while True:
        display_menu("Employee")
        try:
            choice = input("Enter choice: ").strip()
            if choice == "0":
                return "exit"
            elif choice == "5":
                return "logout"
            elif choice == "1":
                start = input("Start date (YYYY-MM-DD): ")
                end = input("End date (YYYY-MM-DD): ")
                reason = input("Reason: ")
                if request_leave(user["id"], user["name"], start, end, reason):
                    print("✅ Leave request submitted!")
                else:
                    print("❌ Failed to submit leave")
            elif choice == "2":
                filename = input("Filename: ")
                desc = input("Description: ")
                if upload_document(user["id"], user["name"], filename, desc):
                    print("✅ Document uploaded!")
            elif choice == "3":
                leaves = search_records(LEAVES_FILE, "employee_id", user["id"])
                print(f"\nYour leave requests ({len(leaves)}):")
                for leave in leaves:
                    print(f"  [{leave['status']}] {leave['start_date']} to {leave['end_date']} - {leave['reason']}")
            elif choice == "4":
                anns = load_data(ANNOUNCEMENTS_FILE)
                print(f"\nAnnouncements ({len(anns)}):")
                for ann in reversed(anns):
                    print(f"  📢 {ann['title']} ({ann['date']})")
                    print(f"     {ann['content'][:80]}...")
            else:
                print("❌ Invalid choice")
        except Exception as e:
            print(f"❌ Error: {e}")

def hr_menu(user):
    """HR interactive menu."""
    while True:
        display_menu("HR")
        try:
            choice = input("Enter choice: ").strip()
            if choice == "0":
                return "exit"
            elif choice == "6":
                return "logout"
            elif choice == "1":
                pending = generate_report("pending_leaves")
                if not pending:
                    print("✅ No pending leaves")
                else:
                    print("\nPending Leaves:")
                    for i, leave in enumerate(pending, 1):
                        print(f"{i}. {leave['employee_name']}: {leave['start_date']} to {leave['end_date']}")
                        print(f"   Reason: {leave['reason']}")
                    try:
                        idx = int(input("Enter number to process (0 to skip): ")) - 1
                        if 0 <= idx < len(pending):
                            action = input("Approve (A) or Reject (R): ").upper()
                            status = "Approved" if action == "A" else "Rejected"
                            update_leave_status(pending[idx]["id"], status)
                            print(f"✅ Leave {status}")
                    except ValueError:
                        print("❌ Invalid input")
            elif choice == "2":
                title = input("Announcement title: ")
                content = input("Content: ")
                if post_announcement(title, content, user["name"]):
                    print("✅ Announcement posted!")
            elif choice == "3":
                docs = load_data(DOCUMENTS_FILE)
                print(f"\nAll Documents ({len(docs)}):")
                for doc in docs:
                    print(f"  - {doc['employee_name']}: {doc['filename']} ({doc['description']})")
            elif choice == "4":
                print("1. Pending Leaves  2. All Documents  3. Announcements")
                rtype = input("Choose report: ")
                mapping = {"1": "pending_leaves", "2": "all_documents", "3": "announcements"}
                report = generate_report(mapping.get(rtype, "pending_leaves"))
                print(f"\nReport: {len(report)} records")
                for item in report:
                    print(f"  - {item}")
            elif choice == "5":
                name = input("Name: ")
                username = input("Username: ")
                password = input("Password: ")
                dept = input("Department: ")
                email = input("Email: ")
                role = input("Role (Employee/HR): ")
                record = {
                    "username": username,
                    "password": password,
                    "name": name,
                    "role": role,
                    "department": dept,
                    "email": email
                }
                if add_record(EMPLOYEES_FILE, record):
                    print("✅ Employee added!")
            else:
                print("❌ Invalid choice")
        except Exception as e:
            print(f"❌ Error: {e}")

def main():
    """Main portal loop with while True."""
    print("\n" + "=" * 50)
    print("   HR MANAGEMENT PORTAL")
    print("=" * 50)
    
    while True:  # Main loop as required
        try:
            print("\n1. Login")
            print("0. Exit")
            choice = input("Choice: ").strip()
            
            if choice == "0":
                print("👋 Goodbye!")
                break
            elif choice == "1":
                username = input("Username: ")
                password = input("Password: ")
                user = user_login(username, password)
                
                if user:
                    print(f"\n✅ Welcome, {user['name']}!")
                    if user["role"] == "Employee":
                        result = employee_menu(user)
                    else:
                        result = hr_menu(user)
                    
                    if result == "exit":
                        print("👋 Goodbye!")
                        break
                    # else logout, continue to login
                else:
                    print("❌ Invalid credentials")
            else:
                print("❌ Invalid choice")
        except KeyboardInterrupt:
            print("\n👋 Interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"❌ Unexpected error: {e}")

# Uncomment the line below to start the interactive portal
# main()
print("\n💡 Interactive portal ready. Uncomment main() and run to start.")
print("   Or use the Streamlit frontend: streamlit run app.py")


## Summary

This HR Portal fully satisfies the assignment requirements:

✅ **Modular functions** with docstrings, parameters, and return values  
✅ **while True** main loop + **for loops** for data processing  
✅ **JSON file persistence** with read on startup, write on updates  
✅ **try-except-finally** for FileNotFoundError, PermissionError, ValueError, JSON errors  
✅ Clear console UI + optional Streamlit frontend  

### How to Run
1. **Console**: Uncomment `main()` in the last cell and run
2. **Frontend**: `streamlit run app.py` (install streamlit first)
3. Data persists in `data/` folder

### Files
- `hr_portal.ipynb` - This notebook
- `app.py` - Streamlit frontend
- `data/*.json` - Persistent storage
- `docs/` - Uploaded documents (Streamlit)
